# 🥔 Potato Disease Grad-CAM Analysis - Colab

This notebook tests Grad-CAM visualization and disease severity quantification across all three models:
- **CNN Baseline** (92.58% accuracy)
- **Transfer Learning / ResNet50V2** (97.40% accuracy)
- **MobileNetV2** (99.61% accuracy)

**Runtime:** GPU recommended (Runtime → Change runtime type → T4 GPU)

In [ ]:
#@title 1. Install Dependencies and Clone Repo
#@markdown Run this cell first to set up the environment
!pip install -q tensorflow Pillow scipy matplotlib numpy

# Clone the repository (or upload files manually)
!git clone https://github.com/AswinPanta/potato-disease-classification.git 2>/dev/null || echo "Repo already cloned or using uploaded files"

import os
os.chdir('potato-disease-classification')
!ls -la saved_models/

In [ ]:
#@title 2. Import Libraries and Load Models
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
import time
from pathlib import Path

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Class names
CLASS_NAMES = ["Early Blight", "Late Blight", "Healthy"]
IMAGE_SIZE = 256

# Model paths
MODEL_PATHS = {
    'cnn-baseline': 'saved_models/1',
    'transfer-learning': 'saved_models/2',
    'mobilenetv2': 'saved_models/3'
}

# MobileNetV2-specific imports
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.resnet_v2 import preprocess_input as resnet_preprocess

In [ ]:
#@title 3. Grad-CAM Utility Functions

CONV_LAYER_TYPES = (
    tf.keras.layers.Conv2D,
    tf.keras.layers.DepthwiseConv2D,
    tf.keras.layers.SeparableConv2D,
)

def find_last_conv_layer(model, model_id=None):
    """Find the best conv layer for Grad-CAM."""
    if model_id == 'mobilenetv2':
        # Skip depthwise, find Conv2D >= 14x14
        candidates = []
        for layer in model.layers:
            if isinstance(layer, tf.keras.layers.Conv2D):
                shape = layer.output_shape
                if len(shape) == 4:
                    h, w = shape[1], shape[2]
                    if h is not None and w is not None and h >= 14 and w >= 14:
                        candidates.append((layer.name, layer, h * w))
        if candidates:
            candidates.sort(key=lambda c: c[2])
            return candidates[0][0]
    
    # Default: find deepest conv layer
    best_name = None
    best_depth = -1
    
    def search(layer, depth=0):
        nonlocal best_name, best_depth
        if isinstance(layer, CONV_LAYER_TYPES):
            if depth >= best_depth:
                best_name = layer.name
                best_depth = depth
        elif hasattr(layer, 'layers'):
            for sub in layer.layers:
                search(sub, depth + 1)
    
    for layer in model.layers:
        search(layer, 0)
    return best_name

def find_layer_by_name(model, name):
    """Find layer by name in model."""
    for layer in model.layers:
        if layer.name == name:
            return layer
        if hasattr(layer, 'layers'):
            result = find_layer_by_name(layer, name)
            if result:
                return result
    return None

def compute_gradcam(model, image, model_id):
    """Compute Grad-CAM heatmap."""
    layer_name = find_last_conv_layer(model, model_id)
    layer = find_layer_by_name(model, layer_name)
    
    img_tensor = tf.cast(image, tf.float32)
    
    grad_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=[layer.output, model.output]
    )
    
    with tf.GradientTape() as tape:
        tape.watch(img_tensor)
        conv_outputs, predictions = grad_model(img_tensor)
        predicted_class = tf.argmax(predictions[0])
        loss = predictions[:, predicted_class]
    
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs), axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + tf.keras.backend.epsilon())
    
    return heatmap.numpy(), predictions[0].numpy()

def create_overlay(image, heatmap):
    """Create heatmap overlay on image."""
    import matplotlib.cm as cm
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], (IMAGE_SIZE, IMAGE_SIZE)).numpy().squeeze()
    heatmap_gamma = np.power(np.clip(heatmap_resized, 0, 1), 0.7)
    colored = cm.jet(heatmap_gamma)[:, :, :3]
    colored_255 = (colored * 255).astype(np.uint8)
    overlay = (image.astype(np.float32) * 0.6 + colored_255.astype(np.float32) * 0.4)
    return np.clip(overlay, 0, 255).astype(np.uint8)

def create_mask(heatmap, threshold=0.5):
    """Create binary mask from heatmap."""
    from scipy.ndimage import binary_opening, binary_closing
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], (IMAGE_SIZE, IMAGE_SIZE)).numpy().squeeze()
    mask = (heatmap_resized > threshold).astype(np.uint8)
    mask = binary_opening(mask, iterations=2)
    mask = binary_closing(mask, iterations=2)
    return mask.astype(np.uint8) * 255

def compute_severity(mask, heatmap):
    """Compute disease severity metrics."""
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], (IMAGE_SIZE, IMAGE_SIZE)).numpy().squeeze()
    total = mask.shape[0] * mask.shape[1]
    affected = np.sum(mask > 0)
    pct = (affected / total) * 100
    
    if affected > 0:
        mean_intensity = float(np.mean(heatmap_resized[mask > 0]))
    else:
        mean_intensity = 0.0
    
    if pct < 8:
        level = "Healthy"
    elif pct < 20:
        level = "Mild"
    elif pct < 50:
        level = "Moderate"
    else:
        level = "Severe"
    
    return {
        'affected_pct': round(pct, 2),
        'mean_intensity': round(mean_intensity, 4),
        'severity_level': level
    }

print("✓ Grad-CAM utilities loaded!")

In [ ]:
#@title 4. Load All Models
models = {}

# CNN Baseline
try:
    models['cnn-baseline'] = tf.keras.models.load_model(MODEL_PATHS['cnn-baseline'])
    print(f"✓ CNN Baseline loaded ({models['cnn-baseline'].count_params():,} params)")
except Exception as e:
    print(f"✗ CNN Baseline failed: {e}")

# Transfer Learning (ResNet50V2)
try:
    models['transfer-learning'] = tf.keras.models.load_model(MODEL_PATHS['transfer-learning'])
    print(f"✓ Transfer Learning loaded ({models['transfer-learning'].count_params():,} params)")
except Exception as e:
    print(f"✗ Transfer Learning failed: {e}")

# MobileNetV2
try:
    models['mobilenetv2'] = tf.keras.models.load_model(MODEL_PATHS['mobilenetv2'])
    print(f"✓ MobileNetV2 loaded ({models['mobilenetv2'].count_params():,} params)")
except Exception as e:
    print(f"✗ MobileNetV2 failed: {e}")

print(f"\nLoaded {len(models)} models: {list(models.keys())}")

## 5. Test with Sample Images

Upload your own potato leaf images or use the generated test patterns below.

In [ ]:
#@title 5a. Generate Test Patterns (or upload your own images)
np.random.seed(42)

def create_disease_pattern(disease_type="early"):
    """Create synthetic disease pattern for testing."""
    img = np.zeros((256, 256, 3), dtype=np.uint8)
    
    if disease_type == "early":
        # Brown spots pattern
        img[:,:,1] = np.clip(np.random.randint(60, 130, (256, 256)), 0, 255)
        img[:,:,0] = np.clip(np.random.randint(40, 90, (256, 256)), 0, 255)
        img[:,:,2] = np.clip(np.random.randint(20, 50, (256, 256)), 0, 255)
        for _ in range(12):
            cx, cy = np.random.randint(40, 216, 2)
            r = np.random.randint(10, 30)
            y, x = np.ogrid[-cx:256-cx, -cy:256-cy]
            mask = x*x + y*y <= r*r
            img[mask] = [110, 75, 35]
    
    elif disease_type == "late":
        # Dark lesion pattern
        img[:,:,1] = np.clip(np.random.randint(50, 110, (256, 256)), 0, 255)
        img[:,:,0] = np.clip(np.random.randint(30, 70, (256, 256)), 0, 255)
        img[:,:,2] = np.clip(np.random.randint(20, 45, (256, 256)), 0, 255)
        for _ in range(8):
            cx, cy = np.random.randint(50, 206, 2)
            r = np.random.randint(20, 45)
            y, x = np.ogrid[-cx:256-cx, -cy:256-cy]
            mask = x*x + y*y <= r*r
            img[mask] = [35, 45, 25]
    
    else:  # healthy
        img[:,:,1] = np.clip(np.random.randint(110, 190, (256, 256)), 0, 255)
        img[:,:,0] = np.clip(np.random.randint(35, 75, (256, 256)), 0, 255)
        img[:,:,2] = np.clip(np.random.randint(25, 55, (256, 256)), 0, 255)
    
    return img

# Create test images
test_images = {
    "Early Blight Pattern": create_disease_pattern("early"),
    "Late Blight Pattern": create_disease_pattern("late"),
    "Healthy Pattern": create_disease_pattern("healthy")
}

# Visualize test images
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, img) in zip(axes, test_images.items()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis('off')
plt.suptitle("Test Images", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
#@title 5b. Upload Your Own Images (Optional)
from google.colab import files
import io

uploaded = files.upload()

if uploaded:
    for filename, data in uploaded.items():
        img = Image.open(io.BytesIO(data)).convert('RGB').resize((256, 256))
        test_images[f"Uploaded: {filename}"] = np.array(img)
        print(f"✓ Loaded: {filename}")
else:
    print("No files uploaded, using generated test patterns.")

print(f"\nTotal test images: {len(test_images)}")

## 6. Run Grad-CAM Analysis on All Models

In [ ]:
#@title 6. Analyze All Models
import matplotlib.cm as cm

def preprocess_image(image, model_id):
    """Preprocess image for specific model."""
    if model_id == 'mobilenetv2':
        return mobilenet_preprocess(image)
    elif model_id == 'transfer-learning':
        return resnet_preprocess(image)
    return image.astype(np.float32)

# Run analysis
all_results = {}

for img_name, img in test_images.items():
    print(f"\n{'='*70}")
    print(f"Image: {img_name}")
    print(f"{'='*70}")
    
    img_results = {}
    
    for model_id, model in models.items():
        print(f"\n  Model: {model_id}")
        
        # Find Grad-CAM layer
        layer_name = find_last_conv_layer(model, model_id)
        print(f"    Grad-CAM Layer: {layer_name}")
        
        # Preprocess
        processed = preprocess_image(np.expand_dims(img, 0), model_id)
        
        # Predict
        start = time.time()
        pred = model.predict(processed, verbose=0)[0]
        pred_time = (time.time() - start) * 1000
        
        pred_class = CLASS_NAMES[np.argmax(pred)]
        confidence = float(np.max(pred))
        
        # Grad-CAM
        start = time.time()
        heatmap, _ = compute_gradcam(model, processed, model_id)
        gradcam_time = (time.time() - start) * 1000
        
        # Severity
        mask = create_mask(heatmap, threshold=0.5)
        severity = compute_severity(mask, heatmap)
        
        print(f"    Prediction: {pred_class} ({confidence:.1%})")
        print(f"    Probabilities: {', '.join([f'{c}: {p:.1%}' for c, p in zip(CLASS_NAMES, pred)])}")
        print(f"    Severity: {severity['severity_level']} ({severity['affected_pct']:.1f}% affected)")
        print(f"    Times: pred={pred_time:.1f}ms, gradcam={gradcam_time:.1f}ms")
        
        img_results[model_id] = {
            'prediction': pred_class,
            'confidence': confidence,
            'probabilities': pred,
            'heatmap': heatmap,
            'mask': mask,
            'severity': severity,
            'layer': layer_name
        }
    
    all_results[img_name] = img_results

## 7. Visualize Results

In [ ]:
#@title 7. Grad-CAM Visualization for Each Image
for img_name, img in test_images.items():
    fig, axes = plt.subplots(2, len(models), figsize=(5*len(models), 10))
    fig.suptitle(f"Analysis: {img_name}", fontsize=16, fontweight='bold')
    
    for col, (model_id, result) in enumerate(all_results[img_name].items()):
        # Top row: Overlay
        overlay = create_overlay(img, result['heatmap'])
        axes[0, col].imshow(overlay)
        axes[0, col].set_title(
            f"{model_id}\n{result['prediction']} ({result['confidence']:.1%})",
            fontsize=10
        )
        axes[0, col].axis('off')
        
        # Bottom row: Masked
        masked = img.copy()
        mask_binary = result['mask'] > 0
        masked[~mask_binary] = [255, 255, 255]
        axes[1, col].imshow(masked)
        axes[1, col].set_title(
            f"Severity: {result['severity']['severity_level']}\n"
            f"Affected: {result['severity']['affected_pct']:.1f}%",
            fontsize=10
        )
        axes[1, col].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
#@title 8. Summary Table
print("\n" + "=" * 90)
print("GRAD-CAM ANALYSIS SUMMARY")
print("=" * 90)

print(f"\n{'Image':<25} {'Model':<20} {'Prediction':<15} {'Confidence':<12} {'Severity':<12} {'Affected':<10}")
print("-" * 94)

for img_name, img_results in all_results.items():
    for model_id, result in img_results.items():
        print(
            f"{img_name:<25} "
            f"{model_id:<20} "
            f"{result['prediction']:<15} "
            f"{result['confidence']:<12.1%} "
            f"{result['severity']['severity_level']:<12} "
            f"{result['severity']['affected_pct']:<10.1f}"
        )
    print()

print("=" * 90)

## 9. Model Accuracy Comparison

In [ ]:
#@title 9. Model Performance Metrics
print("\n" + "=" * 70)
print("MODEL ARCHITECTURE COMPARISON")
print("=" * 70)

model_info = {
    'cnn-baseline': {
        'name': 'CNN Baseline',
        'accuracy': '92.58%',
        'params': '232K',
        'architecture': '6-Conv Custom CNN'
    },
    'transfer-learning': {
        'name': 'Transfer Learning',
        'accuracy': '97.40%',
        'params': '24.6M',
        'architecture': 'ResNet50V2 + Dense'
    },
    'mobilenetv2': {
        'name': 'MobileNetV2',
        'accuracy': '99.61%',
        'params': '2.4M',
        'architecture': 'MobileNetV2 + Dense'
    }
}

for model_id, info in model_info.items():
    print(f"\n{info['name']}:")
    print(f"  Architecture: {info['architecture']}")
    print(f"  Parameters: {info['params']}")
    print(f"  Test Accuracy: {info['accuracy']}")
    if model_id in models:
        print(f"  Grad-CAM Layer: {all_results[list(all_results.keys())[0]][model_id]['layer']}")

print("\n" + "=" * 70)

## 10. Save Results to Drive (Optional)

In [ ]:
#@title 10. Save Results to Google Drive
#@markdown Mount Google Drive to save results
from google.colab import drive
drive.mount('/content/drive')

import json
import os

# Prepare results for saving
save_results = {}
for img_name, img_results in all_results.items():
    save_results[img_name] = {}
    for model_id, result in img_results.items():
        save_results[img_name][model_id] = {
            'prediction': result['prediction'],
            'confidence': result['confidence'],
            'severity': result['severity'],
            'layer': result['layer']
        }

# Save to Drive
save_path = '/content/drive/MyDrive/gradcam_results.json'
with open(save_path, 'w') as f:
    json.dump(save_results, f, indent=2)

print(f"✓ Results saved to {save_path}")

## Key Findings

### Grad-CAM Quality by Model
| Model | Grad-CAM Quality | Best For |
|-------|------------------|----------|
| CNN Baseline | ✅ Good | Quick visualization, low compute |
| Transfer Learning (ResNet50V2) | ✅ Excellent | Most consistent heatmaps |
| MobileNetV2 | ⚠️ Variable | Fast inference, mobile deployment |

### MobileNetV2 Notes
- Uses `block_6_project` as Grad-CAM layer (Conv2D with ≥14×14 spatial dims)
- Skips depthwise convolutions for better gradient flow
- May produce less detailed heatmaps than ResNet50V2
- Best suited for deployment, not visualization

### Recommendations
1. **For visualization**: Use Transfer Learning (ResNet50V2) model
2. **For deployment**: Use MobileNetV2 (fastest, smallest)
3. **For accuracy**: All models perform well (>92%), MobileNetV2 leads at 99.61%